# MSENet: GeM pooling and auxiliary supervised contrastive loss (4-way ablation)

**How to run:** upload this notebook to Kaggle, enable a **GPU** accelerator and **Internet**, then click **Run All**. Nothing else is needed (MiniImageNet is downloaded from the shared Drive folder inside the notebook).

| # | Configuration | Pooling | Loss |
|---|---|---|---|
| 1 | `baseline` | GAP (unchanged) | episodic cross-entropy |
| 2 | `gem` | **GeM**, one learnable exponent `p` per stage (init 3) | episodic cross-entropy |
| 3 | `contrastive` | GAP (unchanged) | cross-entropy **+ lambda x supervised contrastive** (learnable temperature) |
| 4 | `gem_contrastive` | **GeM** | cross-entropy **+ lambda x supervised contrastive** |

Everything else is the baseline notebook, unchanged: ResNet-18 backbone and 5 stage taps, `Self_Attn` modules, learnable stage weights, learnable gamma, Euclidean distance, Adam + CosineAnnealingLR, AMP, `RandomHorizontalFlip` augmentation, 30-way 5-shot 15-query training, **10 epochs x 360 episodes** for every configuration.

**Design decisions that the baseline did not define (all configurable in the "Experiment configuration" cell):**
* *GeM*: `p = softplus(raw) + 0.01` (always positive), initialised to exactly 3, one exponent per stage. **Where it is applied:** it replaces *only* the `AdaptiveAvgPool2d` that follows each `Self_Attn` module, i.e. it pools the attention-residual output `gamma * attn(x) + x` at **all 5 stages, for both support and query features** (in `_attend_and_pool`). Nothing before the pooling (backbone, attention) and nothing after it (prototypes, Euclidean distance, stage weights) changes; the pooled vector keeps the same dimension as with GAP.
  **Handling of negative activations:** `gamma*attn(x)` can be negative, so `x + gamma*attn(x)` is not guaranteed non-negative (the ReLU feature `x` is). Instead of clamping negatives to eps (which would discard them), GeM uses a *signed* (positive/negative split) representation: `GeM(a) = GeM_p(relu(a)) - GeM_p(relu(-a))`, where each branch is a standard GeM of a non-negative tensor (computed in float32 via log-sum-exp, eps = 1e-6 only guards `log(0)`). For `p = 1` this is exactly global average pooling (`mean(a)`), so GeM is a strict generalisation of the baseline pooling; for `p > 1` it emphasises the strongest positive and the strongest negative responses. Positive, negative and near-zero activation fractions are measured over **all stages and all test episodes** and saved in `final_parameters.json`.
* *Contrastive loss*: supervised contrastive (SupCon) over the 150 support + 450 query pooled embeddings of the episode, with their episode labels, computed **per stage** on an L2-normalised *copy* of the pooled feature (the features used for the Euclidean distance are untouched) and averaged over the 5 stages. One shared learnable temperature `tau = exp(log_tau)`, init 0.1, clamped to [0.01, 1]. Coefficient `CON_COEF = 0.1`. Cross-entropy is kept as is.
* The new scalar parameters (`gem_raw`, `con_log_tau`) get their own Adam groups with lr 1e-3 (same scheduler as everything else).
* *Pretraining consistency*: the ImageNet weights are downloaded **once**, and every configuration builds its backbone from that same state dict. If `USE_IMAGENET_PRETRAINED=True` and the download fails, the notebook stops with an error instead of silently falling back to a random backbone for one configuration. Before training, each backbone is verified tensor-by-tensor against the pretrained weights, and initial backbone/attention checksums must match across the four configurations.
* *Fair evaluation*: validation / test / benchmark episodes are drawn with fixed seeds, so **all four models are evaluated on exactly the same episodes**. This is verified with a BLAKE2b digest of every episode (support images, support labels, query images, query labels, shapes, dtypes), checked across configurations, and the digest function is self-tested (deterministic, seed-sensitive, sensitive to a one-pixel or label change). The model used for testing is the best-validation epoch, as in the baseline. The training episode stream is also seeded identically for the four runs.
* *Speed (no change to the experiment)*: every image is decoded and resized **once** (multi-threaded) and cached in RAM as a `uint8` tensor (about 1.3 GB for all three splits at 84x84). Episodes are then gathered from this cache by the unchanged `TaskSampler`; `ToTensor`/`Normalize` (/255, mean/std) and the training-time `RandomHorizontalFlip(p=0.5)` (independent per image) are applied on the GPU, which gives the same pixel values as the original PIL transforms. Training already runs under AMP (`autocast` + `GradScaler`, unchanged); evaluation and inference timing stay in fp32 exactly as in the baseline protocol. Per-step `.item()` GPU synchronisations in the training loop were replaced by on-GPU accumulation (identical numbers, no stalls).
* *RAM (Kaggle limit 29 GB)*: the uint8 image cache (about 1.3 GB) is the only large host object; there is no `.pth` saving, only one best-epoch state dict (about 45 MB) is kept, memory is freed between configurations (`gc` + `malloc_trim`), and the process RSS is logged and guarded every epoch.
* Test accuracy is reported for 5-way 5-shot and 5-way 1-shot (the model is trained once, with 5-shot episodes, as in the paper) over 600 episodes each; the 95% CI is `1.96 * std / sqrt(600)` over per-episode accuracies.

Expected runtime on a Kaggle T4: dominated by the GPU compute (the JPEG decoding that used to run in every episode is now a one-time cost of a few minutes). Results of each finished configuration are cached, so if the session dies you can re-run and finished configurations are skipped.

In [ ]:
# ============ Cell 1: warnings, installs, MiniImageNet download + extraction ============
import os, sys, subprocess, warnings, tarfile
from pathlib import Path

# Non-critical warnings are silenced; exceptions are never caught/hidden.
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working").resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)

def _pip_install(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise RuntimeError(f"pip install failed for {pkgs}")

_pip_install("easyfsl==1.5.0", "gdown")   # same easyfsl version as the baseline run

DRIVE_FOLDER = "https://drive.google.com/drive/folders/17a09kkqVivZQFggCw9I_YboJ23tcexNM"
DOWNLOAD_DIR = WORK_DIR / "miniimagenet_archives"
DATA_DIR = WORK_DIR / "miniimagenet"
EXPECTED_CLASSES = {"train": 64, "val": 16, "test": 20}

def _split_ready(split):
    d = DATA_DIR / split / split
    return d.is_dir() and sum(1 for p in d.iterdir() if p.is_dir()) == EXPECTED_CLASSES[split]

if all(_split_ready(s) for s in EXPECTED_CLASSES):
    print("MiniImageNet already extracted; skipping download.")
else:
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for attempt in range(1, 4):
        # Download the files in the shared Drive folder
        result = subprocess.run([sys.executable, "-m", "gdown", "--folder", DRIVE_FOLDER, "-O", str(DOWNLOAD_DIR)], check=False)
        print("Download command exit code:", result.returncode, f"(attempt {attempt}/3)")
        if all(list(DOWNLOAD_DIR.rglob(f"{s}.tar")) for s in EXPECTED_CLASSES):
            break
    print("Downloaded files:")
    for p in DOWNLOAD_DIR.rglob("*"):
        if p.is_file():
            print(f"{p.name}: {p.stat().st_size / 1e6:.1f} MB")

    # Find and extract train, val, and test archives
    for split in ("train", "val", "test"):
        matches = list(DOWNLOAD_DIR.rglob(f"{split}.tar"))
        if not matches:
            raise FileNotFoundError(
                f"{split}.tar was not found. Check that the Drive folder "
                "is publicly accessible, Internet is enabled, and the download completed."
            )
        archive = matches[0]
        out_dir = DATA_DIR / split
        out_dir.mkdir(parents=True, exist_ok=True)
        print(f"\nExtracting {archive.name} into {out_dir} ...")
        with tarfile.open(archive, "r:*") as tar:
            try:
                tar.extractall(out_dir, filter="data")
            except TypeError:          # older Python without the `filter` argument
                tar.extractall(out_dir)
        print(f"Finished extracting {split}")

    print("\nTop-level extracted structure:")
    for split_dir in sorted(DATA_DIR.iterdir()):
        if split_dir.is_dir():
            print(split_dir.name, "->", [p.name for p in list(split_dir.iterdir())[:8]])

for s in EXPECTED_CLASSES:
    if not _split_ready(s):
        raise RuntimeError(f"Split '{s}' is not extracted correctly (expected {EXPECTED_CLASSES[s]} class folders in {DATA_DIR / s / s}).")
print("Dataset ready.")

In [ ]:
# ============ Cell 2: environment, imports and configuration (baseline config kept verbatim) ============
import os, time, random, copy, gc, json, math, base64, platform, datetime, zipfile, hashlib, ctypes
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn, optim
from torch.utils.data import DataLoader, Sampler
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.models.feature_extraction import create_feature_extractor, get_graph_node_names
from easyfsl.samplers import TaskSampler
from tqdm.auto import tqdm
import multiprocessing

# Force 'spawn' start method to eliminate Python 3.13 multiprocessing cleanup warnings
try:
  multiprocessing.set_start_method('spawn', force=True)
except RuntimeError:
  pass

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. In Kaggle: Settings -> Accelerator -> GPU (T4/P100), then Run All again.")
device = torch.device('cuda')
print('Device:', device)
print('GPU:', torch.cuda.get_device_name(0))

ROOT_DIRECTORY = str(DATA_DIR)
TRAIN_DIR = f"{ROOT_DIRECTORY}/train/train"
VAL_DIR   = f"{ROOT_DIRECTORY}/val/val"
TEST_DIR  = f"{ROOT_DIRECTORY}/test/test"

# ---------------- Experiment configuration (baseline values, unchanged) ----------------
IMAGE_SIZE = 84
MAX_EPOCHS = 10
N_TRAINING_EPISODES = 360
N_VALIDATION_TASKS = 600
N_TEST_TASKS = 600
TRAIN_WAY = 30
TEST_WAY = 5
SHOT = 5
QUERY = 15
FEATURE_CHUNK_SIZE = 600
ATTENTION_CHUNK_SIZE = 600

DISTANCE_METRIC = 'euclidean'
COSINE_TEMP = 10.0           # baseline parameter (unused by the Euclidean metric); kept for an unchanged baseline
USE_AMP = True

STAGE_NODES   = ('maxpool', 'layer1', 'layer2', 'layer3', 'layer4')
STAGE_WEIGHTS = (1.0, 1.1, 1.2, 1.3, 1.4)

# NOTE: miniImageNet is a subset of ImageNet-1k, so ImageNet-pretrained weights have already
# seen the val/test classes. Set to False for a leakage-free few-shot benchmark.
USE_IMAGENET_PRETRAINED = True
PRELOAD_THREADS = min(16, max(4, (os.cpu_count() or 4) * 2))   # threads for the one-time decode + resize (PIL releases the GIL)

# Baseline preprocessing, unchanged: Resize((IMAGE_SIZE, IMAGE_SIZE)) -> [RandomHorizontalFlip (train only)] -> ToTensor -> Normalize(ImageNet mean/std).
# Resize is applied once at load time (PIL, identical to the baseline transform); flip + /255 + normalise are applied on the GPU per episode
# (see `to_dev`), which is numerically the same operation as ToTensor + Normalize.
resize_transform = transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))
IMG_MEAN = (0.485, 0.456, 0.406)
IMG_STD  = (0.229, 0.224, 0.225)

# baseline optimisation hyper-parameters (from the baseline training cell)
gamma_init = 0.2
weights_lr = 0.005
backbone_lr = 1e-4

# ---------------- Experiment configuration (NEW: only used by configs 2-4) ----------------
# GeM
GEM_P_INIT  = 3.0      # initial exponent p
GEM_EPS     = 1e-6     # clamp for numerical stability
GEM_P_FLOOR = 1e-2     # p = softplus(raw) + floor  -> always strictly positive
GEM_LR      = 1e-3     # Adam lr of the GeM exponents
# Supervised contrastive auxiliary loss
CON_COEF     = 0.1     # tunable loss coefficient (lambda):  loss = CE + CON_COEF * SupCon
CON_TAU_INIT = 0.1     # initial temperature (learnable, always positive: tau = exp(log_tau))
CON_TAU_MIN, CON_TAU_MAX = 0.01, 1.0
CON_TAU_LR   = 1e-3

# Evaluation protocol (fixed seeds -> identical episodes for every configuration)
VAL_SEED, TEST_SEED_5SHOT, TEST_SEED_1SHOT, BENCH_SEED = 2024, 2025, 2026, 2027
BENCH_WARMUP, BENCH_EPISODES = 10, 100
LOG_EVERY = 20                      # trajectory logging interval (training episodes)
RESUME = True                       # skip a configuration if its cached result matches the current settings

RESULT_DIR = WORK_DIR / "msenet_results"      # deliverables (zipped at the end)
CACHE_DIR  = WORK_DIR / "msenet_cache"        # per-configuration cache (not zipped)
RESULT_DIR.mkdir(parents=True, exist_ok=True); CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- RAM management (Kaggle: 29 GB) ----------------
RAM_LIMIT_GB = 29.0
RAM_WARN_FRAC, RAM_FAIL_FRAC = 0.80, 0.92

def rss_gb():
    """Resident set size of this process in GB (Linux /proc)."""
    with open('/proc/self/status') as f:
        for line in f:
            if line.startswith('VmRSS:'):
                return int(line.split()[1]) / 2**20
    return float('nan')

def free_host_memory():
    gc.collect()
    try: ctypes.CDLL('libc.so.6').malloc_trim(0)      # give freed heap pages back to the OS
    except OSError: pass

def ram_guard(tag=''):
    """Log RSS; free memory when above 80% of the limit; stop with a clear error above 92% (before the OOM-killer does)."""
    rss = rss_gb()
    if rss > RAM_WARN_FRAC * RAM_LIMIT_GB:
        free_host_memory(); rss = rss_gb()
        print(f'[RAM] WARNING {tag}: RSS {rss:.1f} GB is above {RAM_WARN_FRAC:.0%} of the {RAM_LIMIT_GB:.0f} GB limit.')
        if rss > RAM_FAIL_FRAC * RAM_LIMIT_GB:
            raise MemoryError(f'Process RSS {rss:.1f} GB exceeds {RAM_FAIL_FRAC:.0%} of the {RAM_LIMIT_GB:.0f} GB Kaggle RAM limit.')
    return rss

print(f'[RAM] start: process RSS = {rss_gb():.2f} GB (limit {RAM_LIMIT_GB:.0f} GB)')
print('Imports and configuration ready.')

In [ ]:
# ============ Cell 3: dataset folder check (baseline) ============
from pathlib import Path

for split, path in [
    ("train", TRAIN_DIR),
    ("val", VAL_DIR),
    ("test", TEST_DIR),
]:
    p = Path(path)
    classes = [d for d in p.iterdir() if d.is_dir()] if p.exists() else []

    print(f"{split}: exists={p.exists()}, class folders={len(classes)}")
    print("Examples:", [d.name for d in classes[:5]])

In [ ]:
# ============ Cell 4: evaluation helpers ============
# --- GPU-side preprocessing of cached uint8 episodes ---
_mean_gpu = torch.tensor(IMG_MEAN, dtype=torch.float32).view(1, 3, 1, 1)
_std_gpu  = torch.tensor(IMG_STD,  dtype=torch.float32).view(1, 3, 1, 1)

def to_dev(images_u8, train=False):
    """uint8 (N,3,H,W) CPU tensor -> normalised float32 CUDA tensor.
    Same maths as torchvision ToTensor (x/255) + Normalize ((x-mean)/std); with train=True also RandomHorizontalFlip(p=0.5), drawn independently per image."""
    x = images_u8.pin_memory().to(device, non_blocking=True).float().div_(255.0)
    if train:
        flip = torch.rand(x.shape[0], device=device) < 0.5
        x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    return x.sub_(_mean_gpu.to(device)).div_(_std_gpu.to(device))

# --- baseline helpers ---
@torch.no_grad()
def evaluate_on_one_task(model, support_images, support_labels, query_images, query_labels):
    scores = model(to_dev(support_images), support_labels.to(device), to_dev(query_images))
    predictions = scores.argmax(dim=1)
    labels = query_labels.to(device)
    return (predictions == labels).sum().item(), labels.numel()

@torch.no_grad()
def evaluate(data_loader, model, desc='Evaluating'):
    model.eval()
    correct_total, prediction_total, n_tasks = 0, 0, 0
    for support_images, support_labels, query_images, query_labels, _ in tqdm(data_loader, total=len(data_loader), desc=desc, leave=False):
        correct, total = evaluate_on_one_task(model, support_images, support_labels, query_images, query_labels)
        correct_total += correct
        prediction_total += total
        n_tasks += 1
    if prediction_total == 0:
        raise RuntimeError('Evaluation loader produced no query examples.')
    if n_tasks != len(data_loader):
        raise RuntimeError(f'Evaluated {n_tasks} tasks but loader promised {len(data_loader)}.')
    return 100.0 * correct_total / prediction_total

# --- harness additions (no effect on training): fixed-seed episodes + per-episode metrics ---
def episode_digest(support_images, support_labels, query_images, query_labels):
    # Content hash of a whole episode: pixels AND labels of support and query, shapes/dtypes, order-sensitive (BLAKE2b, 128 bit).
    h = hashlib.blake2b(digest_size=16)
    for t in (support_images, support_labels, query_images, query_labels):
        a = t.detach().cpu().contiguous().numpy()
        h.update(repr((a.dtype.str, a.shape)).encode()); h.update(a)
    return h.hexdigest()

class fixed_rng:
    """Draw evaluation episodes from a fixed seed, then restore the previous RNG state so the
    training episode stream is not disturbed by evaluation."""
    def __init__(self, seed): self.seed = seed
    def __enter__(self):
        self.py = random.getstate(); self.np = np.random.get_state(); self.t = torch.get_rng_state()
        self.cu = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        random.seed(self.seed); np.random.seed(self.seed); torch.manual_seed(self.seed)
    def __exit__(self, *exc):
        random.setstate(self.py); np.random.set_state(self.np); torch.set_rng_state(self.t)
        if self.cu is not None: torch.cuda.set_rng_state_all(self.cu)
        return False

@torch.no_grad()
def evaluate_full(data_loader, model, seed, desc='Evaluating'):
    """Same forward/argmax as `evaluate`, but also returns the cross-entropy loss and per-episode accuracies."""
    model.eval()
    task_acc, task_loss, task_digest = [], [], []
    with fixed_rng(seed):
        for support_images, support_labels, query_images, query_labels, _ in tqdm(data_loader, total=len(data_loader), desc=desc, leave=False):
            task_digest.append(episode_digest(support_images, support_labels, query_images, query_labels))   # episode identity
            scores = model(to_dev(support_images), support_labels.to(device), to_dev(query_images))
            labels = query_labels.to(device)
            task_loss.append(F.cross_entropy(scores.float(), labels).item())
            task_acc.append((scores.argmax(dim=1) == labels).float().mean().item())
    if len(task_acc) != len(data_loader):
        raise RuntimeError(f'Evaluated {len(task_acc)} tasks but loader promised {len(data_loader)}.')
    task_acc = 100.0 * np.array(task_acc)
    return dict(acc=float(task_acc.mean()), loss=float(np.mean(task_loss)), task_acc=task_acc, task_digest=task_digest)

In [ ]:
# ============ Cell 5: BASELINE model classes (verbatim from baseline.ipynb, untouched) ============
class Self_Attn(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        if in_dim < 8:
            raise ValueError(f'in_dim must be >= 8, got {in_dim}')
        self.query_conv = nn.Conv2d(in_dim, in_dim // 8, kernel_size=1)
        self.key_conv = nn.Conv2d(in_dim, in_dim // 8, kernel_size=1)
        self.value_conv = nn.Conv2d(in_dim, in_dim, kernel_size=1)

    def forward(self, x, gamma):
        b, c, h, w = x.shape
        n = h * w
        q = self.query_conv(x).reshape(b, -1, n).transpose(1, 2)  # B,N,C/8
        k = self.key_conv(x).reshape(b, -1, n)                   # B,C/8,N
        attention = torch.softmax(torch.bmm(q, k), dim=-1)        # B,N,N
        v = self.value_conv(x).reshape(b, c, n)                   # B,C,N
        out = torch.bmm(v, attention.transpose(1, 2)).reshape(b, c, h, w)
        return gamma * out + x

class PrototypicalNetworks(nn.Module):
    def __init__(self, backbone, image_size, stage_nodes=STAGE_NODES, weight=STAGE_WEIGHTS,
                 weightlearnable=True, gammalearnable=True, gamma1=0.2, gamma2=0.2,
                 feature_chunk_size=600, attention_chunk_size=600,
                 distance_metric='cosine', cosine_temp=10.0):
        super().__init__()
        if len(weight) != len(stage_nodes):
            raise ValueError(f'{len(weight)} weights given for {len(stage_nodes)} stages.')
        self.weightlearnable = weightlearnable
        self.gammalearnable = gammalearnable
        self.feature_chunk_size = feature_chunk_size
        self.attention_chunk_size = attention_chunk_size
        self.n_stages = len(stage_nodes)
        self.distance_metric = distance_metric

        # Learnable temperature scalar for cosine distance scaling
        self.cosine_temp = nn.Parameter(torch.tensor(float(cosine_temp)), requires_grad=True)

        self.stage_weights = nn.Parameter(torch.tensor(list(weight), dtype=torch.float32),
                                          requires_grad=weightlearnable)
        self.gamma1_l = nn.Parameter(torch.tensor(float(gamma1)), requires_grad=gammalearnable)
        self.gamma2_l = nn.Parameter(torch.tensor(float(gamma2)), requires_grad=gammalearnable)

        graph_nodes, _ = get_graph_node_names(backbone)
        return_nodes = {}
        for i, name in enumerate(stage_nodes):
            matches = [n for n in graph_nodes if n == name or n.startswith(name + '.')]
            if not matches:
                raise ValueError(f'Stage {name!r} not found in backbone graph.')
            return_nodes[matches[-1]] = f'stage_{i}'
        print('Stage nodes:', return_nodes)

        self.feature_extractor = create_feature_extractor(backbone, return_nodes=return_nodes)

        was_training = self.feature_extractor.training
        self.feature_extractor.eval()
        with torch.no_grad():
            dummy = self.feature_extractor(torch.zeros(1, 3, image_size, image_size))
        self.feature_extractor.train(was_training)
        channels = [dummy[f'stage_{i}'].shape[1] for i in range(self.n_stages)]
        print('Selected feature channels:', channels,
              '| spatial sizes:', [tuple(dummy[f'stage_{i}'].shape[2:]) for i in range(self.n_stages)])

        self.attention_modules = nn.ModuleList([Self_Attn(c) for c in channels])
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)

    def _extract_in_chunks(self, images):
        # Fast path: if batch fits in chunk_size, skip Python loop & dict rebuilding!
        if images.size(0) <= self.feature_chunk_size:
            return self.feature_extractor(images)
        collected = None
        for chunk in images.split(self.feature_chunk_size, dim=0):
            feats = self.feature_extractor(chunk)
            if collected is None:
                collected = {key: [value] for key, value in feats.items()}
            else:
                for key, value in feats.items(): collected[key].append(value)
        return {key: torch.cat(values, dim=0) for key, values in collected.items()}

    def _attend_and_pool(self, features, stage_idx, gamma):
        module = self.attention_modules[stage_idx]
        # Fast path: skip chunking loop when batch size <= attention_chunk_size
        if features.size(0) <= self.attention_chunk_size:
            return self.global_avg_pool(module(features, gamma)).flatten(1)
        pooled = []
        for chunk in features.split(self.attention_chunk_size, dim=0):
            pooled.append(self.global_avg_pool(module(chunk, gamma)).flatten(1))
        return torch.cat(pooled, dim=0)

    def _stage_distance(self, support_features, query_features, support_labels, stage_idx):
        z_support = self._attend_and_pool(support_features, stage_idx, self.gamma1_l)
        z_query = self._attend_and_pool(query_features, stage_idx, self.gamma2_l)

        # Vectorized prototype computation (faster than Python list comprehension over 30 labels)
        labels = torch.unique(support_labels, sorted=True)
        prototypes = torch.stack([z_support[support_labels == label].mean(0) for label in labels])

        if self.distance_metric == 'euclidean':
            return torch.cdist(z_query, prototypes, p=2)

        elif self.distance_metric == 'cosine':
            # L2-normalize query and prototype vectors onto unit hypersphere
            z_q_norm = nn.functional.normalize(z_query, p=2, dim=-1)
            proto_norm = nn.functional.normalize(prototypes, p=2, dim=-1)
            # Cosine distance = (1 - cosine_similarity) scaled by temperature
            cos_sim = torch.mm(z_q_norm, proto_norm.t())
            return self.cosine_temp * (1.0 - cos_sim)

        elif self.distance_metric == 'hybrid':
            # Combines L2-normalized Euclidean distance and Cosine distance
            euc_dist = torch.cdist(z_query, prototypes, p=2)
            z_q_norm = nn.functional.normalize(z_query, p=2, dim=-1)
            proto_norm = nn.functional.normalize(prototypes, p=2, dim=-1)
            cos_dist = self.cosine_temp * (1.0 - torch.mm(z_q_norm, proto_norm.t()))
            return 0.5 * euc_dist + 0.5 * cos_dist
        else:
            raise ValueError(f"Unknown distance_metric: {self.distance_metric}")

    def forward(self, support_images, support_labels, query_images):
        support_feats = self._extract_in_chunks(support_images)
        query_feats = self._extract_in_chunks(query_images)
        distances = torch.stack([
            self._stage_distance(support_feats[f'stage_{i}'], query_feats[f'stage_{i}'], support_labels, i)
            for i in range(self.n_stages)
        ])
        return -(self.stage_weights.view(-1, 1, 1) * distances).sum(dim=0)

print('Model class defined.')

In [ ]:
# ============ Cell 6: the ONLY two additions: GeM pooling and supervised contrastive loss ============
def _gem_nonneg(t, p, eps):
    # Plain GeM of a NON-NEGATIVE map t (B,C,H,W): (mean(max(t,eps)^p))^(1/p) via log-sum-exp (float32, stable for any p > 0).
    logt = t.clamp(min=eps).log().flatten(2)                                   # B,C,N
    return torch.exp((torch.logsumexp(p * logt, dim=-1) - math.log(logt.shape[-1])) / p)

def gem_pool(x, p, eps=GEM_EPS):
    """Signed GeM over H,W for activations that can be negative (the attention residual y = gamma*attn + x).
    Split y = y+ - y-  with  y+ = max(y,0), y- = max(-y,0)  (both non-negative), then
        out = GeM_p(y+) - GeM_p(y-).
    - keeps the channel dimension and the sign information (negatives are NOT clamped to eps),
    - p = 1 gives exactly GAP:  mean(y+) - mean(y-) = mean(y),
    - the eps floors of the two branches cancel for ~0 channels.
    x: (B,C,H,W), p: positive scalar tensor  ->  (B,C) in x.dtype (the dtype the GAP path produced)."""
    with torch.autocast(device_type='cuda', enabled=False):
        xf, pf = x.float(), p.float()
        out = _gem_nonneg(xf, pf, eps) - _gem_nonneg(-xf, pf, eps)
    return out.to(x.dtype)

def supcon_loss(z, labels, tau):
    """Supervised contrastive loss (Khosla et al., 2020) over all rows of z (anchors = every sample,
    positives = other samples with the same label). z is L2-normalised here."""
    z = F.normalize(z.float(), dim=1)
    logits = z @ z.t() / tau
    logits = logits - logits.max(dim=1, keepdim=True)[0].detach()
    n = z.size(0)
    not_self = ~torch.eye(n, dtype=torch.bool, device=z.device)
    pos = (labels.view(-1, 1) == labels.view(1, -1)) & not_self
    exp_logits = torch.exp(logits) * not_self
    log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-12)
    n_pos = pos.sum(dim=1)
    valid = n_pos > 0
    mean_log_prob_pos = (log_prob * pos).sum(dim=1)[valid] / n_pos[valid]
    return -mean_log_prob_pos.mean()

class MSENetVariant(PrototypicalNetworks):
    """Baseline model + optional GeM pooling (one exponent per stage) + optional contrastive-loss head.
    Distance, stage weights, gamma, backbone and attention modules are inherited unchanged.
    WHERE GeM IS APPLIED: only in `_attend_and_pool`, replacing `global_avg_pool` on the output of each of the
    5 `Self_Attn` modules (y = gamma*attn + x), for support and query features, before prototypes / Euclidean distance."""
    def __init__(self, *args, use_gem=False, use_contrastive=False, **kwargs):
        super().__init__(*args, **kwargs)
        self.use_gem = use_gem
        self.use_contrastive = use_contrastive
        self.capture = False          # when True, pooled embeddings are stored for the contrastive loss
        self._emb = {}
        self._diag = None             # activation statistics, only accumulated while enabled (test evaluation)
        if use_gem:
            raw0 = math.log(math.expm1(GEM_P_INIT - GEM_P_FLOOR))     # softplus(raw0) + floor == GEM_P_INIT exactly
            self.gem_raw = nn.Parameter(torch.full((self.n_stages,), raw0, dtype=torch.float32))
        if use_contrastive:
            self.con_log_tau = nn.Parameter(torch.tensor(math.log(CON_TAU_INIT), dtype=torch.float32))

    @property
    def gem_p(self):                                   # strictly positive, one per stage
        return F.softplus(self.gem_raw) + GEM_P_FLOOR

    @property
    def con_tau(self):                                 # strictly positive, bounded
        return torch.exp(self.con_log_tau.clamp(math.log(CON_TAU_MIN), math.log(CON_TAU_MAX)))

    def reset_diag(self):
        # start accumulating attention-output statistics for ALL stages and ALL following batches (support + query)
        z = lambda: torch.zeros((), dtype=torch.float64, device=self.gem_raw.device)
        self._diag = {i: dict(n_all=z(), n_neg=z(), n_small=z(), abs_all=z(), abs_neg=z()) for i in range(self.n_stages)}

    def read_diag(self):
        d, self._diag = self._diag, None
        out, tot = {}, defaultdict(float)
        for i in range(self.n_stages):
            v = {k: float(t) for k, t in d[i].items()}
            for k, x in v.items(): tot[k] += x
            out[f'stage_{i+1}'] = dict(n_activations=int(v['n_all']), frac_negative=v['n_neg'] / v['n_all'],
                                       frac_abs_below_eps=v['n_small'] / v['n_all'], negative_abs_mass_share=v['abs_neg'] / v['abs_all'])
        out['all_stages'] = dict(n_activations=int(tot['n_all']), frac_negative=tot['n_neg'] / tot['n_all'],
                                 frac_abs_below_eps=tot['n_small'] / tot['n_all'], negative_abs_mass_share=tot['abs_neg'] / tot['abs_all'])
        return out

    def _gem_stage(self, features, stage_idx, gamma):
        module = self.attention_modules[stage_idx]
        p = self.gem_p[stage_idx]
        outs = []
        for chunk in features.split(self.attention_chunk_size, dim=0):
            a = module(chunk, gamma)                                        # y = gamma*attn + x  (can be negative)
            if self._diag is not None:
                with torch.no_grad():
                    d = self._diag[stage_idx]
                    d['n_all'] += a.numel(); d['n_neg'] += (a < 0).sum(); d['n_small'] += (a.abs() < GEM_EPS).sum()
                    d['abs_all'] += a.abs().sum(dtype=torch.float64); d['abs_neg'] += (-a).clamp(min=0).sum(dtype=torch.float64)
            outs.append(gem_pool(a, p))
        return outs[0] if len(outs) == 1 else torch.cat(outs, dim=0)

    def _attend_and_pool(self, features, stage_idx, gamma):
        if self.use_gem:
            z = self._gem_stage(features, stage_idx, gamma)                # GeM replaces GAP
        else:
            z = super()._attend_and_pool(features, stage_idx, gamma)       # original GAP path
        if self.capture:
            self._emb[(stage_idx, 's' if gamma is self.gamma1_l else 'q')] = z
        return z

    def contrastive_loss(self, support_labels, query_labels):
        """SupCon over support+query pooled embeddings of the current episode, averaged over the 5 stages."""
        labels = torch.cat([support_labels, query_labels])
        tau = self.con_tau.float()
        total = 0.0
        with torch.autocast(device_type='cuda', enabled=False):
            for i in range(self.n_stages):
                z = torch.cat([self._emb[(i, 's')], self._emb[(i, 'q')]], dim=0)
                total = total + supcon_loss(z, labels, tau)
        self._emb = {}
        return total / self.n_stages

print('GeM (signed) / contrastive extensions defined.')

In [ ]:
# ============ Cell 7: Fast episode sampler compatibility patch (verbatim) ============
# Fast episode sampler compatibility patch for EasyFSL / newer PyTorch
# The patch uses precomputed labels; it never loads image files to discover labels.
def _patched_task_sampler_init(self, dataset, n_way, n_shot, n_query, n_tasks):
    Sampler.__init__(self)
    self.n_way, self.n_shot, self.n_query, self.n_tasks = n_way, n_shot, n_query, n_tasks
    self.items_per_label = defaultdict(list)
    for item, label in enumerate(dataset.get_labels()):
        self.items_per_label[int(label)].append(item)
    if len(self.items_per_label) < n_way:
        raise ValueError(f'Requested {n_way}-way episodes but dataset has only {len(self.items_per_label)} classes.')
    required = n_shot + n_query
    too_small = [label for label, items in self.items_per_label.items() if len(items) < required]
    if too_small:
        raise ValueError(f'Classes have fewer than {required} images: {too_small[:5]}')

TaskSampler.__init__ = _patched_task_sampler_init
print('TaskSampler patched; labels are read from metadata only.')

In [ ]:
# ============ Cell 8: datasets and episode loaders (baseline + 1-shot test and benchmark loaders) ============
class PreloadedSet:
    """All images of one split, decoded + resized ONCE and kept in RAM as a uint8 tensor (N,3,H,W).
    File listing, class order and labels come from torchvision's ImageFolder (as in the baseline); pixels are identical to
    ImageFolder's loader (PIL open -> convert('RGB')) followed by transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))."""
    def __init__(self, root, name):
        folder = ImageFolder(root)                                   # no transform: only used to list files / labels
        self.classes, self.targets = folder.classes, list(folder.targets)
        paths = [pth for pth, _ in folder.samples]
        n = len(paths)
        self.images = torch.empty((n, 3, IMAGE_SIZE, IMAGE_SIZE), dtype=torch.uint8)
        view = self.images.numpy()                                   # shares memory; every thread writes its own slice
        def load(i):
            with Image.open(paths[i]) as im:
                im = resize_transform(im.convert('RGB'))
            view[i] = np.asarray(im, dtype=np.uint8).transpose(2, 0, 1)
        t0 = time.time()
        with ThreadPoolExecutor(max_workers=PRELOAD_THREADS) as pool:
            for _ in tqdm(pool.map(load, range(n)), total=n, desc=f'Preloading {name}', leave=False):
                pass                                                 # exceptions of any image are re-raised here (never hidden)
        self.labels = torch.tensor(self.targets, dtype=torch.long)
        print(f"{name}: {n} images preloaded in {time.time() - t0:.0f}s -> {self.images.numel() / 2**20:.0f} MB uint8 in RAM")
    def __len__(self): return len(self.targets)
    def get_labels(self): return self.targets                        # EasyFSL sampler compatibility

train_set = PreloadedSet(TRAIN_DIR, 'train')
val_set   = PreloadedSet(VAL_DIR,   'val')
test_set  = PreloadedSet(TEST_DIR,  'test')
free_host_memory(); print(f'[RAM] after preloading all splits: {rss_gb():.2f} GB')

print(f"Classes: train={len(train_set.classes)}, val={len(val_set.classes)}, test={len(test_set.classes)}")
print(f"Images: train={len(train_set)}, val={len(val_set)}, test={len(test_set)}")

assert len(train_set.classes) >= TRAIN_WAY, "Not enough training classes for TRAIN_WAY."
assert len(val_set.classes) >= TEST_WAY, "Not enough validation classes for TEST_WAY."
assert len(test_set.classes) >= TEST_WAY, "Not enough test classes for TEST_WAY."

class EpisodeLoader:
    """Episodes from a PreloadedSet. The episode *indices* come from the unchanged EasyFSL TaskSampler (same sampling rule, same python RNG,
    so `fixed_rng` still gives identical episodes); the collation replicates `TaskSampler.episodic_collate_fn` (same support/query split,
    same label re-mapping and class-id order) but gathers from the RAM cache with one indexing op instead of decoding 600 JPEGs.
    Yields (support_u8, support_labels, query_u8, query_labels, class_ids); images are uint8 CPU tensors -> `to_dev` on the GPU."""
    def __init__(self, store, n_way, n_tasks, shot, query):
        self.store, self.n_way, self.shot, self.query = store, n_way, shot, query
        self.sampler = TaskSampler(store, n_way=n_way, n_shot=shot, n_query=query, n_tasks=n_tasks)
    def __len__(self): return len(self.sampler)
    def collate(self, idx):
        idx_t = torch.as_tensor(idx, dtype=torch.long)
        lab = self.store.labels[idx_t].tolist()
        class_ids = list({x for x in lab})                           # identical construction (and order) as EasyFSL
        remap = {c: i for i, c in enumerate(class_ids)}
        labels = torch.tensor([remap[c] for c in lab]).reshape(self.n_way, self.shot + self.query)
        imgs = self.store.images[idx_t].reshape(self.n_way, self.shot + self.query, 3, IMAGE_SIZE, IMAGE_SIZE)
        return (imgs[:, :self.shot].reshape(-1, 3, IMAGE_SIZE, IMAGE_SIZE), labels[:, :self.shot].flatten(),
                imgs[:, self.shot:].reshape(-1, 3, IMAGE_SIZE, IMAGE_SIZE), labels[:, self.shot:].flatten(), class_ids)
    def __iter__(self):
        for idx in self.sampler:
            yield self.collate(idx)

def make_loader(dataset, n_way, n_tasks, shot=SHOT, query=QUERY):
    loader = EpisodeLoader(dataset, n_way, n_tasks, shot, query)
    assert len(loader) == n_tasks, f"Loader has {len(loader)} tasks, expected {n_tasks}."
    return loader

train_loader      = make_loader(train_set, TRAIN_WAY, N_TRAINING_EPISODES)
val_loader        = make_loader(val_set,   TEST_WAY,  N_VALIDATION_TASKS)
test_loader       = make_loader(test_set,  TEST_WAY,  N_TEST_TASKS)                  # 5-way 5-shot
test_loader_1shot = make_loader(test_set,  TEST_WAY,  N_TEST_TASKS, shot=1)          # 5-way 1-shot
bench_loader      = make_loader(test_set,  TEST_WAY,  BENCH_WARMUP + BENCH_EPISODES) # inference timing (5-way 5-shot)
print(f"Episodes per epoch: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}")

# Smoke test: verify the episode shapes really follow the configuration.
def check_episode(loader, way, shot=SHOT):
    s_img, s_lab, q_img, q_lab, _ = next(iter(loader))
    assert s_img.dtype == torch.uint8 and q_img.dtype == torch.uint8
    assert s_img.shape == (way * shot, 3, IMAGE_SIZE, IMAGE_SIZE), s_img.shape
    assert q_img.shape == (way * QUERY, 3, IMAGE_SIZE, IMAGE_SIZE), q_img.shape
    assert len(s_lab) == way * shot and len(q_lab) == way * QUERY
    assert len(set(s_lab.tolist())) == way
    print(f"{way}-way {shot}-shot episode OK:", tuple(s_img.shape), tuple(q_img.shape))

# The fast collate must equal EasyFSL's episodic_collate_fn, and the GPU preprocessing must equal torchvision's ToTensor + Normalize.
_ref_norm = transforms.Compose([transforms.ToTensor(), transforms.Normalize(IMG_MEAN, IMG_STD)])
with fixed_rng(123):
    _idx = next(iter(val_loader.sampler))
_S, _Sl, _Q, _Ql, _ids = val_loader.collate(_idx)
_ref = val_loader.sampler.episodic_collate_fn([(_ref_norm(Image.fromarray(val_set.images[i].permute(1, 2, 0).contiguous().numpy())), val_set.targets[i]) for i in _idx])
assert torch.equal(_Sl, _ref[1]) and torch.equal(_Ql, _ref[3]) and _ids == _ref[4], 'Fast collate differs from EasyFSL labels / class ids.'
assert (to_dev(_S).cpu() - _ref[0]).abs().max() < 1e-5 and (to_dev(_Q).cpu() - _ref[2]).abs().max() < 1e-5, 'GPU preprocessing differs from ToTensor + Normalize.'
_a = to_dev(_Q[:64], train=True).cpu(); _b = to_dev(_Q[:64]).cpu()
assert all(torch.equal(_a[i], _b[i]) or torch.equal(_a[i], _b[i].flip(2)) for i in range(64)), 'Training augmentation is not a pure horizontal flip.'
del _S, _Sl, _Q, _Ql, _ref, _a, _b, _idx
print('Preloaded pipeline check OK: same episodes / labels as EasyFSL, same pixels as ToTensor + Normalize, flip augmentation is a pure h-flip.')

t0 = time.time()
check_episode(train_loader, TRAIN_WAY)
check_episode(val_loader, TEST_WAY)
check_episode(test_loader_1shot, TEST_WAY, shot=1)
print(f"Smoke test done in {time.time() - t0:.2f}s")

In [ ]:
# ============ Cell 8b: sanity check of the episode digest used to prove identical evaluation episodes ============
def _first_digests(loader, seed, n=5):
    out = []
    with fixed_rng(seed):
        for i, (S, Sl, Q, Ql, _) in enumerate(loader):
            out.append(episode_digest(S, Sl, Q, Ql))
            if i + 1 == n: break
    return out

_d1 = _first_digests(test_loader, TEST_SEED_5SHOT)
_d2 = _first_digests(test_loader, TEST_SEED_5SHOT)
_d3 = _first_digests(test_loader, TEST_SEED_5SHOT + 1)
assert _d1 == _d2, 'Digest is not reproducible for the same seed: episodes are NOT deterministic.'
assert len(set(_d1)) == len(_d1) and not set(_d1) & set(_d3), 'Digest does not separate different episodes.'
with fixed_rng(TEST_SEED_5SHOT):
    _S, _Sl, _Q, _Ql, _ = next(iter(test_loader))
_base = episode_digest(_S, _Sl, _Q, _Ql)
_Q2 = _Q.clone(); _Q2.view(-1)[0] = 255 - _Q2.view(-1)[0]                  # one-pixel change (uint8 pixels)
_Ql2 = _Ql.clone()
_unique = torch.unique(_Ql)
_Ql2[0] = _unique[_unique != _Ql[0]][0]                                    # change one query label to a different class
_Sl2 = _Sl.flip(0)                                                         # reorder support labels
assert len({_base, episode_digest(_S, _Sl, _Q2, _Ql), episode_digest(_S, _Sl, _Q, _Ql2), episode_digest(_S, _Sl2, _Q, _Ql)}) == 4, 'Digest is not sensitive.'
del _S, _Sl, _Q, _Ql, _Q2, _Ql2, _Sl2, _unique
print('Episode digest OK: reproducible for a fixed seed, different across seeds, sensitive to pixel / label / order changes.')
free_host_memory(); print(f'[RAM] after dataset setup: {rss_gb():.2f} GB')

In [ ]:
# ============ Cell 9: experiment runner (baseline training loop + logging) ============
EXPERIMENTS = {
    'baseline':        dict(label='1. Baseline',             use_gem=False, use_con=False),
    'gem':             dict(label='2. GeM only',             use_gem=True,  use_con=False),
    'contrastive':     dict(label='3. Contrastive only',     use_gem=False, use_con=True),
    'gem_contrastive': dict(label='4. GeM + Contrastive',    use_gem=True,  use_con=True),
}
ORDER = list(EXPERIMENTS)

FINGERPRINT = json.dumps(dict(v=3, data_pipeline='preloaded-uint8-gpu-aug', gem_variant='signed-posneg-split', epochs=MAX_EPOCHS, episodes=N_TRAINING_EPISODES, val=N_VALIDATION_TASKS, test=N_TEST_TASKS,
                              gem=[GEM_P_INIT, GEM_EPS, GEM_P_FLOOR, GEM_LR], con=[CON_COEF, CON_TAU_INIT, CON_TAU_MIN, CON_TAU_MAX, CON_TAU_LR],
                              seeds=[SEED, VAL_SEED, TEST_SEED_5SHOT, TEST_SEED_1SHOT, BENCH_SEED], amp=USE_AMP,
                              pretrained=USE_IMAGENET_PRETRAINED), sort_keys=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def load_pretrained_state():
    # The ImageNet weights are fetched ONCE, before any training. There is no silent fallback: if the download
    # fails the notebook stops, so no configuration can end up with a different initialisation than the others.
    if not USE_IMAGENET_PRETRAINED:
        return None
    sd = resnet18(weights=ResNet18_Weights.DEFAULT).state_dict()
    return {k: v.detach().clone() for k, v in sd.items()}

PRETRAINED_STATE = load_pretrained_state()
print('ImageNet-pretrained backbone:', 'ENABLED for all four configurations' if PRETRAINED_STATE is not None else 'DISABLED for all four configurations (random init)')

def state_digest(*modules):
    h = hashlib.blake2b(digest_size=16)
    for m in modules:
        for k, v in sorted(m.state_dict().items()):
            h.update(k.encode()); h.update(v.detach().cpu().contiguous().numpy())
    return h.hexdigest()

def build_model(use_gem, use_con):
    print('Initializing ResNet-18...')
    backbone = resnet18(weights=None)
    if PRETRAINED_STATE is not None:
        backbone.load_state_dict(PRETRAINED_STATE)
        for k, v in backbone.state_dict().items():          # verify tensor-by-tensor: really the pretrained weights
            assert torch.equal(v, PRETRAINED_STATE[k]), f'pretrained weight mismatch in {k}'
    backbone.fc = nn.Identity()
    common = dict(
        backbone=backbone, image_size=IMAGE_SIZE, stage_nodes=STAGE_NODES, weight=STAGE_WEIGHTS,
        weightlearnable=True, gammalearnable=True, gamma1=gamma_init, gamma2=gamma_init,
        feature_chunk_size=FEATURE_CHUNK_SIZE, attention_chunk_size=ATTENTION_CHUNK_SIZE,
        distance_metric=DISTANCE_METRIC, cosine_temp=COSINE_TEMP,
    )
    if not use_gem and not use_con:
        model = PrototypicalNetworks(**common)                      # the untouched baseline class
    else:
        model = MSENetVariant(**common, use_gem=use_gem, use_contrastive=use_con)
    return model.to(device)

def snapshot_params(model):
    d = {f'stage_w{i+1}': v for i, v in enumerate(model.stage_weights.detach().cpu().tolist())}
    d['gamma1'] = float(model.gamma1_l.detach().cpu()); d['gamma2'] = float(model.gamma2_l.detach().cpu())
    if hasattr(model, 'gem_raw'):
        for i, v in enumerate(model.gem_p.detach().cpu().tolist()): d[f'gem_p{i+1}'] = v
    if hasattr(model, 'con_log_tau'):
        d['con_tau'] = float(model.con_tau.detach().cpu())
    return d

def summarize(arr):
    arr = np.asarray(arr, dtype=np.float64); n = len(arr)
    std = float(arr.std(ddof=1))
    return dict(mean=float(arr.mean()), std=std, ci95=float(1.96 * std / math.sqrt(n)), n=int(n))

@torch.no_grad()
def benchmark_inference(model):
    """fp32 inference (as in the baseline's evaluate), 5-way 5-shot, 15 queries/class; loader time excluded."""
    model.eval()
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    times = []
    with fixed_rng(BENCH_SEED):
        for i, (S, Sl, Q, Ql, _) in enumerate(bench_loader):
            S, Sl, Q = to_dev(S), Sl.to(device), to_dev(Q)
            torch.cuda.synchronize(); t0 = time.perf_counter()
            model(S, Sl, Q)
            torch.cuda.synchronize(); dt = time.perf_counter() - t0
            if i >= BENCH_WARMUP: times.append(dt)
    peak_mb = torch.cuda.max_memory_allocated() / 2**20
    return np.array(times), peak_mb

@torch.no_grad()
def count_gflops(model):
    try:
        from torch.utils.flop_counter import FlopCounterMode
        model.eval()
        with fixed_rng(BENCH_SEED):
            S, Sl, Q, Ql, _ = next(iter(bench_loader))
        with FlopCounterMode(display=False) as fc:
            model(to_dev(S), Sl.to(device), to_dev(Q))
        return fc.get_total_flops() / 1e9
    except Exception as exc:
        print(f'WARNING: FLOP counting failed ({exc!r}); GFLOPs set to NaN.')
        return float('nan')

def run_experiment(key):
    spec = EXPERIMENTS[key]; use_gem, use_con = spec['use_gem'], spec['use_con']
    cache = CACHE_DIR / key / 'result.json'
    if RESUME and cache.exists():
        r = json.loads(cache.read_text())
        if r.get('fingerprint') == FINGERPRINT:
            print(f"[{spec['label']}] cached result found -> skipping training."); return r
    print('=' * 80); print(f"RUNNING {spec['label']}  (GeM={use_gem}, Contrastive={use_con})"); print('=' * 80)

    free_host_memory(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    wall0 = time.time()
    set_seed(SEED)                                        # identical init and identical training episode stream
    model = build_model(use_gem, use_con)
    init_backbone_digest = state_digest(model.feature_extractor)      # must be identical for all four configurations
    init_attention_digest = state_digest(model.attention_modules)
    print(f'[RAM] {key} start: {rss_gb():.2f} GB')

    # ---- optimizer / scheduler / scaler: baseline code, plus one Adam group per NEW scalar parameter ----
    special = [model.stage_weights, model.gamma1_l, model.gamma2_l, model.cosine_temp]
    extra_groups = []
    if use_gem:
        special.append(model.gem_raw);     extra_groups.append({'params': [model.gem_raw], 'lr': GEM_LR})
    if use_con:
        special.append(model.con_log_tau); extra_groups.append({'params': [model.con_log_tau], 'lr': CON_TAU_LR})
    special_ids = {id(p) for p in special}
    regular_params = [p for p in model.parameters() if id(p) not in special_ids]

    param_groups = [
        {'params': regular_params, 'lr': backbone_lr},
        {'params': [model.cosine_temp], 'lr': weights_lr},
    ]
    if model.weightlearnable:
        param_groups.append({'params': [model.stage_weights], 'lr': weights_lr})
    if model.gammalearnable:
        param_groups.append({'params': [model.gamma1_l, model.gamma2_l], 'lr': weights_lr})
    param_groups += extra_groups

    optimizer = optim.Adam(param_groups)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    criterion = nn.CrossEntropyLoss()

    n_total = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in trainable_params)
    n_backbone = sum(p.numel() for p in model.feature_extractor.parameters())
    n_attn = sum(p.numel() for p in model.attention_modules.parameters())
    print(f'Parameters: total={n_total:,} trainable={n_trainable:,} (backbone={n_backbone:,}, attention={n_attn:,})')

    epoch_log, trajectory = [], []
    trajectory.append(dict(step=0, epoch=0, **snapshot_params(model)))
    best_val_acc, best_epoch, best_state = -1.0, 0, None
    global_step = 0
    torch.cuda.reset_peak_memory_stats()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        if use_con: model.capture = True
        # running sums stay on the GPU (float64): no per-step host synchronisation
        loss_sum = torch.zeros((), dtype=torch.float64, device=device); ce_sum = loss_sum.clone(); con_sum = loss_sum.clone(); acc_sum = loss_sum.clone(); n_episodes = 0
        lr_now = optimizer.param_groups[0]['lr']
        torch.cuda.synchronize(); t_train0 = time.time()
        progress = tqdm(train_loader, total=len(train_loader), desc=f'[{key}] Epoch {epoch}/{MAX_EPOCHS}', leave=False)

        for support_images, support_labels, query_images, query_labels, _ in progress:
            optimizer.zero_grad(set_to_none=True)
            sl = support_labels.to(device, non_blocking=True); ql = query_labels.to(device, non_blocking=True)
            S_in = to_dev(support_images, train=True); Q_in = to_dev(query_images, train=True)   # GPU: /255, flip(p=0.5), normalise

            # Wrapped forward pass and loss computation in autocast
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                scores = model(S_in, sl, Q_in)
                ce = criterion(scores, ql)                        # original episodic classification loss (kept)
                if use_con:
                    con = model.contrastive_loss(sl, ql)          # auxiliary supervised contrastive loss
                    loss = ce + CON_COEF * con
                else:
                    con = None
                    loss = ce

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()

            loss_sum += loss.detach().double(); ce_sum += ce.detach().double(); n_episodes += 1
            if con is not None: con_sum += con.detach().double()
            acc_sum += (scores.detach().argmax(dim=1) == ql).float().mean().double()
            global_step += 1
            if global_step % LOG_EVERY == 0:
                trajectory.append(dict(step=global_step, epoch=epoch, **snapshot_params(model)))
            if global_step % LOG_EVERY == 0: progress.set_postfix(loss=f'{loss.item():.4f}')     # sync only every LOG_EVERY steps

        torch.cuda.synchronize(); train_time = time.time() - t_train0
        loss_sum, ce_sum, con_sum, acc_sum = loss_sum.item(), ce_sum.item(), con_sum.item(), acc_sum.item()
        if use_con: model.capture = False; model._emb = {}
        if n_episodes != N_TRAINING_EPISODES:
            raise RuntimeError(f'Ran {n_episodes} episodes, config says {N_TRAINING_EPISODES}. '
                               'Re-run the dataset cell to rebuild the loaders.')

        # Step the learning rate scheduler at the end of every training epoch
        scheduler.step()

        t_val0 = time.time()
        val = evaluate_full(val_loader, model, VAL_SEED, desc=f'[{key}] Validating epoch {epoch}')
        val_time = time.time() - t_val0
        row = dict(config=key, label=spec['label'], epoch=epoch,
                   train_acc=100.0 * acc_sum / n_episodes,
                   train_loss_total=loss_sum / n_episodes, train_loss_ce=ce_sum / n_episodes,
                   train_loss_con=(con_sum / n_episodes) if use_con else float('nan'),
                   val_acc=val['acc'], val_loss=val['loss'], lr_backbone=lr_now,
                   train_time_s=train_time, val_time_s=val_time)
        epoch_log.append(row)
        print(f"[{key}] Epoch {epoch}: train loss(CE)={row['train_loss_ce']:.4f}"
              + (f", con={row['train_loss_con']:.4f}" if use_con else '')
              + f", train acc={row['train_acc']:.2f}%, val loss={val['loss']:.4f}, val accuracy={val['acc']:.2f}%, "
                f"time={(train_time + val_time)/60:.1f} min, RAM={ram_guard(f'{key} epoch {epoch}'):.1f} GB")
        if val['acc'] > best_val_acc:
            best_val_acc, best_epoch = val['acc'], epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}   # in memory only (no .pth files)
            print(f'[{key}] New best validation accuracy (epoch {epoch}).')

    train_peak_alloc = torch.cuda.max_memory_allocated() / 2**20
    train_peak_resv = torch.cuda.max_memory_reserved() / 2**20
    params_end = snapshot_params(model)                  # values at the end of training (epoch 10)
    print(f'[{key}] Best validation accuracy: {best_val_acc:.2f}% (epoch {best_epoch})')

    # ---- final evaluation on held-out test classes, with the best-validation checkpoint (as in the baseline) ----
    model.load_state_dict(best_state)
    params_best = snapshot_params(model)                 # values of the model that is actually tested
    if use_gem: model.reset_diag()                       # activation statistics: all stages, all test batches (5-shot + 1-shot)
    test5 = evaluate_full(test_loader, model, TEST_SEED_5SHOT, desc=f'[{key}] Test 5-way 5-shot')
    test1 = evaluate_full(test_loader_1shot, model, TEST_SEED_1SHOT, desc=f'[{key}] Test 5-way 1-shot')
    gem_diag = model.read_diag() if use_gem else None     # aggregated over all 5 stages and all 1200 test episodes
    print('=' * 48)
    print(f"[{key}] FINAL TEST ACCURACY 5-way 5-shot: {test5['acc']:.2f}% | 5-way 1-shot: {test1['acc']:.2f}%")
    print('=' * 48)
    print('Stage weights (tested model):', [round(params_best[f'stage_w{i+1}'], 4) for i in range(5)])
    print('gamma1/gamma2 (tested model):', round(params_best['gamma1'], 4), round(params_best['gamma2'], 4))
    if use_gem: print('GeM p (tested model):', [round(params_best[f'gem_p{i+1}'], 4) for i in range(5)])
    if use_con: print('Contrastive tau (tested model):', round(params_best['con_tau'], 5))
    if use_gem:
        g = gem_diag['all_stages']
        print(f"Attention-output activations (all stages, all test batches): negative {100*g['frac_negative']:.1f}%, |y|<eps {100*g['frac_abs_below_eps']:.3f}%, "
              f"negative share of |y| mass {100*g['negative_abs_mass_share']:.1f}%")

    # ---- inference-time / memory / FLOPs benchmark ----
    times, infer_peak_mb = benchmark_inference(model)
    gflops = count_gflops(model)
    n_query_imgs = TEST_WAY * QUERY
    train_total = sum(r['train_time_s'] for r in epoch_log); val_total = sum(r['val_time_s'] for r in epoch_log)
    eff = dict(
        params_total=int(n_total), params_trainable=int(n_trainable), params_backbone=int(n_backbone), params_attention=int(n_attn),
        train_time_total_s=train_total, train_time_per_epoch_s=train_total / MAX_EPOCHS, val_time_total_s=val_total,
        wall_time_s=time.time() - wall0,
        infer_ms_per_episode_mean=1000 * float(times.mean()), infer_ms_per_episode_std=1000 * float(times.std(ddof=1)),
        infer_ms_per_episode_median=1000 * float(np.median(times)),
        infer_ms_per_query_image=1000 * float(times.mean()) / n_query_imgs,
        infer_queries_per_second=n_query_imgs / float(times.mean()),
        gflops_per_episode=gflops,
        gpu_train_peak_allocated_mb=train_peak_alloc, gpu_train_peak_reserved_mb=train_peak_resv,
        gpu_infer_peak_allocated_mb=infer_peak_mb,
    )
    result = dict(
        key=key, label=spec['label'], use_gem=use_gem, use_con=use_con, fingerprint=FINGERPRINT,
        best_epoch=best_epoch, best_val_acc=best_val_acc,
        epoch_log=epoch_log, trajectory=trajectory, params_end_of_training=params_end, params_best_checkpoint=params_best,
        gem_activation_diagnostics=gem_diag,
        init_backbone_digest=init_backbone_digest, init_attention_digest=init_attention_digest,
        test={'5shot': dict(**summarize(test5['task_acc']), task_acc=test5['task_acc'].tolist(), digests=test5['task_digest'], loss=test5['loss']),
              '1shot': dict(**summarize(test1['task_acc']), task_acc=test1['task_acc'].tolist(), digests=test1['task_digest'], loss=test1['loss'])},
        efficiency=eff,
    )
    cache.parent.mkdir(parents=True, exist_ok=True)
    cache.write_text(json.dumps(result))

    del model, optimizer, scheduler, scaler, best_state
    free_host_memory(); torch.cuda.empty_cache()
    print(f'[RAM] {key} done: {rss_gb():.2f} GB')
    return result

RESULTS = {}
print('Runner ready.')

In [ ]:
# ============ Configuration 1/4: original baseline (unchanged) ============
RESULTS['baseline'] = run_experiment('baseline')

In [ ]:
# ============ Configuration 2/4: GeM only ============
RESULTS['gem'] = run_experiment('gem')

In [ ]:
# ============ Configuration 3/4: supervised contrastive loss only ============
RESULTS['contrastive'] = run_experiment('contrastive')

In [ ]:
# ============ Configuration 4/4: GeM + supervised contrastive loss ============
RESULTS['gem_contrastive'] = run_experiment('gem_contrastive')

In [ ]:
# ============ Cell 14: comparison tables (CSV) ============
for k in ORDER:
    if k not in RESULTS: raise RuntimeError(f"Missing result for '{k}'. Run the four configuration cells first.")
LABEL = {k: EXPERIMENTS[k]['label'] for k in ORDER}

# integrity: all four configurations must start from the identical pretrained backbone / attention initialisation
for k in ORDER[1:]:
    for f in ('init_backbone_digest', 'init_attention_digest'):
        if RESULTS[k][f] != RESULTS['baseline'][f]:
            raise RuntimeError(f"{f} of '{k}' differs from the baseline: the configurations did not start from the same initialisation.")
print('Initialisation check OK: identical (pretrained =', USE_IMAGENET_PRETRAINED, ') backbone and attention weights in all four configurations.')

# per-epoch metrics
epoch_df = pd.concat([pd.DataFrame(RESULTS[k]['epoch_log']) for k in ORDER], ignore_index=True)
for k in ORDER:
    epoch_df[epoch_df.config == k].to_csv(RESULT_DIR / f'epoch_metrics_{k}.csv', index=False)
epoch_df.to_csv(RESULT_DIR / 'epoch_metrics_all.csv', index=False)

# test accuracy with 95% CI
test_rows = []
for k in ORDER:
    r = RESULTS[k]; t5, t1 = r['test']['5shot'], r['test']['1shot']
    test_rows.append({'config': k, 'label': LABEL[k],
                      'acc_5way_1shot': t1['mean'], 'ci95_5way_1shot': t1['ci95'],
                      'acc_5way_5shot': t5['mean'], 'ci95_5way_5shot': t5['ci95'],
                      'n_test_episodes': t5['n'], 'best_epoch': r['best_epoch'], 'best_val_acc': r['best_val_acc'],
                      'test_ce_loss_5shot': t5['loss']})
test_df = pd.DataFrame(test_rows)
test_df.to_csv(RESULT_DIR / 'test_accuracy.csv', index=False)

# efficiency metrics
eff_rows = []
for k in ORDER:
    e = RESULTS[k]['efficiency']
    eff_rows.append({'config': k, 'label': LABEL[k], **e,
                     'params_extra_vs_baseline': e['params_total'] - RESULTS['baseline']['efficiency']['params_total']})
eff_df = pd.DataFrame(eff_rows)
eff_df.to_csv(RESULT_DIR / 'efficiency_metrics.csv', index=False)

# paired comparison against the baseline (same episodes for all models -> exact equality of per-episode content digests)
paired_rows = []
base_t = RESULTS['baseline']['test']
for k in ORDER[1:]:
    for shot in ('1shot', '5shot'):
        a, b = np.array(RESULTS[k]['test'][shot]['task_acc']), np.array(base_t[shot]['task_acc'])
        da, db = RESULTS[k]['test'][shot]['digests'], base_t[shot]['digests']
        same = bool(len(da) == len(db) == N_TEST_TASKS and da == db)
        if not same:
            print(f'WARNING: test episodes of {k} ({shot}) differ from the baseline episodes; paired CI is not valid.')
        s = summarize(a - b)
        paired_rows.append({'config': k, 'label': LABEL[k], 'setting': f'5-way {shot[0]}-shot', 'mean_diff_vs_baseline': s['mean'],
                            'ci95': s['ci95'], 'ci_excludes_zero': bool(abs(s['mean']) > s['ci95']), 'same_episodes_verified': same})
paired_df = pd.DataFrame(paired_rows)
paired_df.to_csv(RESULT_DIR / 'paired_vs_baseline.csv', index=False)

# per-episode test accuracies (compact)
pe = {'episode': np.arange(N_TEST_TASKS)}
for k in ORDER:
    for shot in ('1shot', '5shot'):
        pe[f'{k}_{shot}'] = np.round(RESULTS[k]['test'][shot]['task_acc'], 4)
pd.DataFrame(pe).to_csv(RESULT_DIR / 'per_episode_test_accuracy.csv', index=False)

# merged, human-readable summary
summary = pd.DataFrame({
    'Configuration': [LABEL[k] for k in ORDER],
    '5-way 1-shot acc (%) +/- 95% CI': [f"{r.acc_5way_1shot:.2f} +/- {r.ci95_5way_1shot:.2f}" for r in test_df.itertuples()],
    '5-way 5-shot acc (%) +/- 95% CI': [f"{r.acc_5way_5shot:.2f} +/- {r.ci95_5way_5shot:.2f}" for r in test_df.itertuples()],
    'Best epoch (val)': test_df.best_epoch.values,
    'Train time (min)': (eff_df.train_time_total_s / 60).round(2).values,
    'Inference (ms/episode)': eff_df.infer_ms_per_episode_mean.round(2).values,
    'Inference (ms/query img)': eff_df.infer_ms_per_query_image.round(3).values,
    'GFLOPs/episode': eff_df.gflops_per_episode.round(2).values,
    'Params total': eff_df.params_total.values, 'Params trainable': eff_df.params_trainable.values,
    'GPU train peak (MB)': eff_df.gpu_train_peak_allocated_mb.round(0).values,
    'GPU train reserved (MB)': eff_df.gpu_train_peak_reserved_mb.round(0).values,
    'GPU infer peak (MB)': eff_df.gpu_infer_peak_allocated_mb.round(0).values,
})
summary.to_csv(RESULT_DIR / 'comparison_summary.csv', index=False)

# parameter trajectories and final parameter reports
traj_df = pd.concat([pd.DataFrame(RESULTS[k]['trajectory']).assign(config=k) for k in ORDER], ignore_index=True)
traj_df.to_csv(RESULT_DIR / 'parameter_trajectories.csv', index=False)

GEM_PLACEMENT = ('Signed GeM replaces only the global average pooling that follows each of the 5 Self_Attn modules '
                 '(input: y = gamma*attn(x) + x, support and query), before prototypes and Euclidean distance. '
                 'pooled = GeM_p(max(y,0)) - GeM_p(max(-y,0)); p=1 equals GAP.')
final = {'_gem_placement': GEM_PLACEMENT}
final.update({k: {'tested_model_best_val_checkpoint': RESULTS[k]['params_best_checkpoint'],
             'end_of_training_epoch_%d' % MAX_EPOCHS: RESULTS[k]['params_end_of_training'],
             'best_epoch': RESULTS[k]['best_epoch'],
             'gem_activation_diagnostics_all_stages_all_test_batches': RESULTS[k]['gem_activation_diagnostics']} for k in ORDER})
(RESULT_DIR / 'final_parameters.json').write_text(json.dumps(final, indent=2))
flat = []
for k in ORDER:
    for which, d in (('tested_model_best_val_checkpoint', RESULTS[k]['params_best_checkpoint']),
                     ('end_of_training', RESULTS[k]['params_end_of_training'])):
        flat.append({'config': k, 'snapshot': which, **d})
pd.DataFrame(flat).to_csv(RESULT_DIR / 'final_parameters.csv', index=False)

run_info = dict(date=datetime.datetime.now().isoformat(timespec='seconds'), python=platform.python_version(), torch=torch.__version__,
                gpu=torch.cuda.get_device_name(0), epochs=MAX_EPOCHS, episodes_per_epoch=N_TRAINING_EPISODES,
                hyperparameters=dict(GEM_P_INIT=GEM_P_INIT, GEM_EPS=GEM_EPS, GEM_P_FLOOR=GEM_P_FLOOR, GEM_LR=GEM_LR, CON_COEF=CON_COEF,
                                     CON_TAU_INIT=CON_TAU_INIT, CON_TAU_MIN=CON_TAU_MIN, CON_TAU_MAX=CON_TAU_MAX, CON_TAU_LR=CON_TAU_LR,
                                     gamma_init=gamma_init, weights_lr=weights_lr, backbone_lr=backbone_lr, STAGE_WEIGHTS=STAGE_WEIGHTS,
                                     DISTANCE_METRIC=DISTANCE_METRIC, USE_AMP=USE_AMP, SEED=SEED),
                gem_placement=GEM_PLACEMENT, imagenet_pretrained_all_configs=USE_IMAGENET_PRETRAINED, process_rss_gb_at_report=round(rss_gb(), 2),
                eval_protocol=dict(episode_identity='BLAKE2b digest of support+query pixels and labels, exact equality across configs', val_episodes=N_VALIDATION_TASKS, test_episodes=N_TEST_TASKS, ci='1.96*std/sqrt(n) over per-episode accuracy',
                                   bench_warmup=BENCH_WARMUP, bench_episodes=BENCH_EPISODES, inference_precision='fp32',
                                   note='GPU memory = torch.cuda.max_memory_allocated; GFLOPs count conv/matmul only'))
(RESULT_DIR / 'run_info.json').write_text(json.dumps(run_info, indent=2))

pd.set_option('display.width', 250); pd.set_option('display.max_columns', 50)
print('Comparison summary'); display(summary)
print('Paired difference vs. baseline (same episodes)'); display(paired_df)
print('Final parameters'); display(pd.DataFrame(flat))
_dg = [dict(config=k, stage=st, **v) for k in ORDER if RESULTS[k]['gem_activation_diagnostics'] for st, v in RESULTS[k]['gem_activation_diagnostics'].items()]
print('GeM input diagnostics (attention output y; all stages, all 1200 test episodes, support + query)'); display(pd.DataFrame(_dg))

In [ ]:
# ============ Cell 15: plots (all saved as PNG in the results folder) ============
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Palette: first four slots of the validated categorical order (color follows the configuration everywhere).
COLORS = {'baseline': '#2a78d6', 'gem': '#eb6834', 'contrastive': '#1baf7a', 'gem_contrastive': '#eda100'}
INK, INK2, MUTED, GRID, AXIS = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
STAGE_RAMP = ['#86b6ef', '#5598e7', '#2a78d6', '#1c5cab', '#104281']      # one-hue ramp: shallow -> deep stage
STAGE_NAMES = ['stage 1 (maxpool)', 'stage 2 (layer1)', 'stage 3 (layer2)', 'stage 4 (layer3)', 'stage 5 (layer4)']
plt.rcParams.update({'font.family': 'sans-serif', 'axes.edgecolor': AXIS, 'axes.labelcolor': INK2, 'xtick.color': INK2, 'ytick.color': INK2,
                     'text.color': INK, 'axes.titlecolor': INK, 'axes.titleweight': 'bold', 'axes.titlesize': 11, 'figure.dpi': 110})
SHORT = {'baseline': 'Baseline', 'gem': 'GeM', 'contrastive': 'Contrastive', 'gem_contrastive': 'GeM + Contrastive'}

def style(ax, grid_axis='y'):
    for s in ('top', 'right'): ax.spines[s].set_visible(False)
    ax.grid(axis=grid_axis, color=GRID, linewidth=0.8); ax.set_axisbelow(True)
    ax.tick_params(length=0)

def save(fig, name):
    fig.savefig(RESULT_DIR / name, dpi=160, bbox_inches='tight', facecolor='white'); plt.show(); plt.close(fig)

xs = np.arange(len(ORDER)); cols = [COLORS[k] for k in ORDER]; names = [SHORT[k] for k in ORDER]

def bar_panel(ax, vals, fmt, ylabel, title, err=None):
    ax.bar(xs, vals, width=0.55, color=cols, yerr=err, error_kw=dict(ecolor=INK2, capsize=4, lw=1.2))
    for x, v in zip(xs, vals): ax.text(x, v + (max(vals) * 0.015), fmt.format(v), ha='center', va='bottom', color=INK, fontsize=9)
    ax.set_xticks(xs); ax.set_xticklabels(names, rotation=15, ha='right'); ax.set_ylabel(ylabel); ax.set_title(title, loc='left')
    ax.set_ylim(0, max(vals) * 1.15 + (0 if err is None else max(err))); style(ax)

# 1) training time
fig, ax = plt.subplots(figsize=(6.5, 4))
bar_panel(ax, (eff_df.train_time_total_s / 60).values, '{:.1f} min', 'minutes', f'Training time ({MAX_EPOCHS} epochs x {N_TRAINING_EPISODES} episodes, validation excluded)')
save(fig, 'plot1_training_time.png')

# 2) inference time
fig, ax = plt.subplots(figsize=(6.5, 4))
bar_panel(ax, eff_df.infer_ms_per_episode_mean.values, '{:.1f} ms', 'ms / episode', 'Inference time (5-way 5-shot, 75 queries, fp32; error bar = std)', err=eff_df.infer_ms_per_episode_std.values)
save(fig, 'plot2_inference_time.png')

# 3) GPU memory
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
bar_panel(axes[0], eff_df.gpu_train_peak_allocated_mb.values, '{:.0f}', 'MB (peak allocated)', 'GPU memory, training (30-way 5-shot)')
bar_panel(axes[1], eff_df.gpu_infer_peak_allocated_mb.values, '{:.0f}', 'MB (peak allocated)', 'GPU memory, inference (5-way 5-shot)')
fig.tight_layout(); save(fig, 'plot3_gpu_memory.png')

# 4) stage-weight / gamma / GeM-p / contrastive-temperature trajectories
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for j, k in enumerate(ORDER):
    ax = axes[0, j]; d = traj_df[traj_df.config == k]
    for s in range(5): ax.plot(d.step, d[f'stage_w{s+1}'], color=STAGE_RAMP[s], lw=2, label=STAGE_NAMES[s])
    ax.set_title(f'Stage weights: {SHORT[k]}', loc='left'); ax.set_xlabel('training episode'); style(ax)
    if j == 0: ax.legend(frameon=False, fontsize=8)
for col, g in ((0, 'gamma1'), (1, 'gamma2')):
    ax = axes[1, col]
    for k in ORDER:
        d = traj_df[traj_df.config == k]; ax.plot(d.step, d[g], color=COLORS[k], lw=2, label=SHORT[k])
    ax.set_title(f'{g} (attention scale)', loc='left'); ax.set_xlabel('training episode'); style(ax)
    if col == 0: ax.legend(frameon=False, fontsize=8)
ax = axes[1, 2]
for k, ls in (('gem', '-'), ('gem_contrastive', '--')):
    d = traj_df[traj_df.config == k]
    for s in range(5): ax.plot(d.step, d[f'gem_p{s+1}'], color=STAGE_RAMP[s], lw=2, ls=ls)
ax.set_title('GeM exponent p per stage', loc='left'); ax.set_xlabel('training episode'); style(ax)
ax.legend(handles=[Line2D([], [], color=INK2, ls='-', label='GeM only'), Line2D([], [], color=INK2, ls='--', label='GeM + Contrastive')] +
          [Line2D([], [], color=STAGE_RAMP[s], label=f'stage {s+1}') for s in range(5)], frameon=False, fontsize=8, ncol=2)
ax = axes[1, 3]
for k in ('contrastive', 'gem_contrastive'):
    d = traj_df[traj_df.config == k]; ax.plot(d.step, d['con_tau'], color=COLORS[k], lw=2, label=SHORT[k])
ax.set_title('Learnable contrastive temperature tau', loc='left'); ax.set_xlabel('training episode'); style(ax); ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); save(fig, 'plot4_parameter_trajectories.png')

# 5) bar plot of test accuracies
fig, ax = plt.subplots(figsize=(9, 4.5)); w = 0.2
for i, k in enumerate(ORDER):
    r = test_df[test_df.config == k].iloc[0]; vals = [r.acc_5way_1shot, r.acc_5way_5shot]
    b = ax.bar(np.arange(2) + (i - 1.5) * w, vals, width=w * 0.9, color=COLORS[k], label=SHORT[k])
    for x, v in zip(np.arange(2) + (i - 1.5) * w, vals): ax.text(x, v + 1, f'{v:.2f}', ha='center', va='bottom', fontsize=8, color=INK)
ax.set_xticks([0, 1]); ax.set_xticklabels(['5-way 1-shot', '5-way 5-shot']); ax.set_ylim(0, 100); ax.set_ylabel('test accuracy (%)')
ax.set_title('Test accuracy on MiniImageNet (600 episodes per setting)', loc='left'); style(ax); ax.legend(frameon=False, ncol=4, loc='upper left', bbox_to_anchor=(0, -0.1))
save(fig, 'plot5_test_accuracy_bar.png')

# 6) test accuracy with 95% CI (dot + interval)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, (col, ci, title) in zip(axes, (('acc_5way_1shot', 'ci95_5way_1shot', '5-way 1-shot'), ('acc_5way_5shot', 'ci95_5way_5shot', '5-way 5-shot'))):
    for i, k in enumerate(ORDER):
        r = test_df[test_df.config == k].iloc[0]
        ax.errorbar(r[col], len(ORDER) - 1 - i, xerr=r[ci], fmt='o', color=COLORS[k], ecolor=COLORS[k], elinewidth=2.5, capsize=5, markersize=8, markeredgecolor='white')
        ax.text(r[col], len(ORDER) - 1 - i + 0.22, f'{r[col]:.2f} +/- {r[ci]:.2f}', ha='center', fontsize=8, color=INK)
    ax.set_yticks(range(len(ORDER))); ax.set_yticklabels([SHORT[k] for k in ORDER[::-1]]); ax.set_ylim(-0.6, len(ORDER) - 0.3)
    ax.set_xlabel('test accuracy (%) with 95% CI'); ax.set_title(title, loc='left'); style(ax, 'x')
fig.tight_layout(); save(fig, 'plot6_test_accuracy_ci.png')

# extra A) learning curves
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for k in ORDER:
    d = epoch_df[epoch_df.config == k]
    for ax, col, t in ((axes[0, 0], 'train_acc', 'Training accuracy (%)'), (axes[0, 1], 'val_acc', 'Validation accuracy (%)'),
                       (axes[1, 0], 'train_loss_ce', 'Training loss (cross-entropy only)'), (axes[1, 1], 'val_loss', 'Validation loss (cross-entropy)')):
        ax.plot(d.epoch, d[col], color=COLORS[k], lw=2, marker='o', ms=4, label=SHORT[k]); ax.set_title(t, loc='left'); ax.set_xlabel('epoch'); style(ax)
axes[0, 1].legend(frameon=False, fontsize=9)
fig.tight_layout(); save(fig, 'extra_learning_curves.png')

# extra B) paired difference vs baseline
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharex=True)
for ax, setting in zip(axes, ('5-way 1-shot', '5-way 5-shot')):
    d = paired_df[paired_df.setting == setting].reset_index(drop=True)
    for i, r in d.iterrows():
        ax.errorbar(r.mean_diff_vs_baseline, len(d) - 1 - i, xerr=r.ci95, fmt='o', color=COLORS[r.config], elinewidth=2.5, capsize=5, markersize=8, markeredgecolor='white')
        ax.text(r.mean_diff_vs_baseline, len(d) - 1 - i + 0.25, f'{r.mean_diff_vs_baseline:+.2f}', ha='center', fontsize=8)
    ax.axvline(0, color=INK2, lw=1, ls='--'); ax.set_yticks(range(len(d))); ax.set_yticklabels([SHORT[c] for c in d.config[::-1]]); ax.set_ylim(-0.6, len(d) - 0.3)
    ax.set_title(f'{setting}: gain over baseline (paired, 95% CI)', loc='left'); ax.set_xlabel('accuracy difference (points)'); style(ax, 'x')
fig.tight_layout(); save(fig, 'extra_paired_difference_vs_baseline.png')

In [ ]:
# ============ Cell 16: ZIP everything and trigger the download ============
from IPython.display import display, HTML, Javascript, FileLink

zip_path = WORK_DIR / 'msenet_four_config_results.zip'
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(RESULT_DIR.rglob('*')):
        if p.is_file(): zf.write(p, arcname=f'msenet_results/{p.relative_to(RESULT_DIR)}')
size_mb = zip_path.stat().st_size / 2**20
print(f'Created {zip_path} ({size_mb:.2f} MB) containing:')
with zipfile.ZipFile(zip_path) as zf:
    for n in zf.namelist(): print('  ', n)

fname = zip_path.name
if size_mb < 25:
    b64 = base64.b64encode(zip_path.read_bytes()).decode()
    js = """
    (function() {
      var a = document.createElement('a');
      a.href = 'data:application/zip;base64,""" + b64 + """';
      a.download = '""" + fname + """';
      document.body.appendChild(a); a.click(); document.body.removeChild(a);
    })();
    """
    display(Javascript(js))                                   # automatic download attempt
    display(HTML(f'<p><a download="{fname}" href="data:application/zip;base64,{b64}">Click here if the download did not start automatically: <b>{fname}</b></a></p>'))
os.chdir(WORK_DIR)
display(FileLink(fname))                                       # second fallback; the ZIP is also in the notebook Output (/kaggle/working)
print('Done.')